In [13]:
import os
import shutil

# 1. Dynamically search for the folder containing train.py
source_dir = None
for root, dirs, files in os.walk("/kaggle/input/"):
    if "train.py" in files:
        source_dir = root
        break

if source_dir:
    print(f"✅ Found project files in: {source_dir}")
    
    # 2. Copy to working directory (since /kaggle/input is read-only)
    dest_dir = "/kaggle/working/cce_project"
    if not os.path.exists(dest_dir):
        shutil.copytree(source_dir, dest_dir)
        print(f"📂 Copied files to {dest_dir}")
    else:
        print(f"📂 Directory {dest_dir} already exists, skipping copy.")
        
    # 3. Change directory
    os.chdir(dest_dir)
    print("✅ Current directory:", os.getcwd())
    
    # 4. Install dependencies
    !pip install -r requirements.txt -q
    print("✅ Setup complete! Ready to train.")
else:
    print("❌ Could not find train.py anywhere in /kaggle/input/")
    print("Debugging contents of /kaggle/input/:")
    !ls -laR /kaggle/input/

✅ Found project files in: /kaggle/input/datasets/govindusrimaan/cce-project-zip
📂 Copied files to /kaggle/working/cce_project
✅ Current directory: /kaggle/working/cce_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 4.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.0/639.0 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.17.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
✅ S

In [24]:
import os

budget_file = "cce/comm/budget.py"
with open(budget_file, "r") as f:
    content = f.read()

# 1. Patch Policy Mode (allow negative lambda for subsidy)
old_policy = "self.lam.clamp_(0.0, self.max_lambda)"
new_policy = "self.lam.clamp_(-1.0, self.max_lambda)  # ALLOW SUBSIDY"

# 2. Patch Utility Mode (allow slightly negative threshold)
old_utility = "self.lam.clamp_(0.0, 1.0)"
new_utility = "self.lam.clamp_(-0.5, 1.0)  # ALLOW SUBSIDY"

success = False
if old_policy in content:
    content = content.replace(old_policy, new_policy)
    success = True
if old_utility in content:
    content = content.replace(old_utility, new_utility)
    success = True

if success:
    with open(budget_file, "w") as f:
        f.write(content)
    print("✅ Successfully patched budget.py on disk! Lambda can now go negative to force communication.")
else:
    print("❌ Could not find the exact text. Debugging lines with 'clamp_':")
    for i, line in enumerate(content.split('\n')):
        if 'clamp_' in line and 'lam' in line:
            print(f"Line {i+1}: {line.strip()}")

✅ Successfully patched budget.py on disk! Lambda can now go negative to force communication.


In [27]:
# ===== COMMUNICATION BOOTCAMP PATCH =====
# --- Patch 1: grid_world.py reward line -> scalable cost + send bonus hook ---
p = "cce/envs/grid_world.py"
s = open(p).read()
old = "- self.cfg.w_comm * comm_cost"
new = ('- getattr(self, "comm_cost_scale", 1.0) * self.cfg.w_comm * comm_cost '
       '+ getattr(self, "send_bonus", 0.0) * comm_cost')
assert old in s, "reward line not found!"
open(p, "w").write(s.replace(old, new))
print("✅ grid_world.py: comm cost scalable + send bonus hook added")

# --- Patch 2: mappo.py train_step -> bootcamp schedule ---
p = "cce/algo/mappo.py"
s = open(p).read()
old = "rollout_stats = self.collect_rollout()"
new = ("self._update_idx = getattr(self, '_update_idx', 0) + 1\n"
       "            warmup = self._update_idx <= 120\n"
       "            self._warmup = warmup\n"
       "            self.env.comm_cost_scale = 0.0 if warmup else 1.0\n"
       "            self.env.send_bonus = 0.5 if warmup else 0.0\n"
       "            rollout_stats = self.collect_rollout()")
assert old in s, "train_step line not found!"
open(p, "w").write(s.replace(old, new))
print("✅ mappo.py: bootcamp schedule wired into train_step")

# --- Diagnostics for the NEXT round (wiring lambda into the send logits) ---
for path, pats in [("cce/algo/networks.py", ["send", "bias", "logit", "budget"]),
                   ("cce/algo/distributions.py", ["send", "logit", "probs", "def "])]:
    print(f"\n========== {path} ==========")
    for i, line in enumerate(open(path).read().split("\n")):
        if any(k in line for k in pats):
            print(f"{i+1:4d}: {line}")

✅ grid_world.py: comm cost scalable + send bonus hook added
✅ mappo.py: bootcamp schedule wired into train_step

========== cce/algo/networks.py ==========
  14: from cce.comm.budget import BudgetController
  23:                               h  →  send head (Bernoulli)
  27:         dist: MultiActionDist over (move, send)
  43:         budget_mode: str = "policy",
  63:         self.send_head = nn.Linear(hidden_dim, 1)
  65:         # Utility head (for utility budget mode)
  66:         if budget_mode == "utility":
  71:         self.budget_controller = BudgetController(
  72:             mode=budget_mode, target_beta=target_beta
  81:                 nn.init.constant_(m.bias, 0)
  97:             dist: MultiActionDist over (move, send)
 105:         move_logits = self.move_head(h)  # (..., 4)
 106:         # Send head with budget bias
 107:         send_logit = self.send_head(h).squeeze(-1)  # (...)
 108:         if self.budget_controller.mode == "policy":
 109:             send_logi

In [29]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 train.py \
    --num_updates 300 \
    --num_envs 32 \
    --grid_size 15 \
    --num_agents 3 \
    --fov_size 5 \
    --max_steps 64 \
    --beta 0.5 \
    --run_dir runs/bootcamp \
    --log_interval 10

Traceback (most recent call last):
  File "/kaggle/working/cce_project/train.py", line 12, in <module>
    from cce.algo.mappo import MAPPO
  File "/kaggle/working/cce_project/cce/algo/__init__.py", line 2, in <module>
    from cce.algo.mappo import MAPPO
  File "/kaggle/working/cce_project/cce/algo/mappo.py", line 410
    warmup = self._update_idx <= 120
IndentationError: unexpected indent
